In [4]:
import pandas
from Dependency.File_converter import selecting
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV,HalvingGridSearchCV

data_training, data_testing = selecting()

In [232]:
#Random_Forest Grid

from sklearn.ensemble import RandomForestClassifier

In [ ]:
#Hist_Grabient_Boosting Grid

from sklearn.ensemble import HistGradientBoostingClassifier

p_grid = {
    "learning_rate":[0.03,0.05,0.1,0.2],
    "max_iter":[80,100,120,140,160],
    "max_leaf_nodes":[20,31,40,50],
    "min_samples_leaf":[10,20,30],
    "l2_regularization":[0,0.05,0.1],
    "max_features":[0.4,0.6,0.8,1.0],
    "categorical_features":['from_dtype',["Day_of_Week","1st_Road_Class","Road_Type","Junction_Control","2nd_Road_Class","Pedestrian_Crossing-Physical_Facilities","Light_Conditions","Weather_Conditions","Road_Surface_Conditions","Urban_or_Rural_Area","Did_Police_Officer_Attend_Scene_of_Accident"]],
    "scoring":['accuracy'],
    "class_weight":[None]
}

H_GB_grid = HalvingGridSearchCV(
    estimator=HistGradientBoostingClassifier(),
    param_grid=p_grid,
    scoring='accuracy',
    n_jobs=2,
    max_resources=500)

data_training_X = data_training.loc[ : , data_training.columns != "Accident_Severity"]
data_training_Y = data_training["Accident_Severity"]

H_GB_grid.fit(data_training_X,data_training_Y)

print("The tuned Categorical Naive Bayes Parameters are : {}".format(H_GB_grid.best_params_))
print("The best score is : {}".format(H_GB_grid.best_score_))


The tuned Categorical Naive Bayes Parameters are : {'categorical_features': 'from_dtype', 'class_weight': None, 'l2_regularization': 0.1, 'learning_rate': 0.05, 'max_features': 1.0, 'max_iter': 100, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy'}
The best score is : 0.46464646464646464


In [11]:
results = pandas.DataFrame(H_GB_grid.cv_results_)

# Worst first
worst = results.sort_values("rank_test_score").iloc[-1]
print(worst["params"], worst["rank_test_score"])

# Compact view of all combos, worst to best
cols = ["params", "mean_test_score", "std_test_score", "rank_test_score"]
print(results[cols].sort_values("rank_test_score").tail(5))

# Best first
best = results.sort_values("rank_test_score").iloc[0]
print(best["params"], best["rank_test_score"])

# Compact view of all combos, worst to best
print(results[cols].sort_values("mean_test_score").head(5))



#results.head()

{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'class_weight': None, 'l2_regularization': 0.1, 'learning_rate': 0.05, 'max_features': 1.0, 'max_iter': 100, 'max_leaf_nodes': 20, 'min_samples_leaf': 30, 'scoring': 'accuracy'} 5
                                              params  mean_test_score  \
4  {'categorical_features': 'from_dtype', 'class_...         0.464646   
0  {'categorical_features': 'from_dtype', 'class_...         0.448485   
1  {'categorical_features': 'from_dtype', 'class_...         0.448485   
3  {'categorical_features': ['Day_of_Week', '1st_...         0.430303   
2  {'categorical_features': ['Day_of_Week', '1st_...         0.430303   

   std_test_score  rank_test_score  
4        0.087361                1  
0        0.110

In [234]:
#Categorical_Naive_Bayes Grid

from sklearn.naive_bayes import CategoricalNB

p_grid = {
    "alpha":[0.5,1.0,1.3,1.5,1.6,1.7,1.8,2.0,3.0],
}

C_NB_grid = GridSearchCV(
    estimator=CategoricalNB(),
    param_grid=p_grid,
    scoring='accuracy',
    n_jobs=2)

def coord_positive(dataset):
    longitude = dataset["Longitude"]
    dataset["Longitude"] = longitude-dataset["Longitude"].min()
    longitude = dataset["Latitude"]
    dataset["Latitude"] = longitude-dataset["Latitude"].min()
    return dataset

data_training_positive = coord_positive(data_training)

data_training_X = data_training_positive.loc[ : , data_training_positive.columns != "Accident_Severity"]
data_training_Y = data_training_positive["Accident_Severity"]

C_NB_grid.fit(data_training_X, data_training_Y)

print("The tuned Categorical Naive Bayes Parameters are : {}".format(C_NB_grid.best_params_))
print("The best score is : {}".format(C_NB_grid.best_score_))

The tuned Categorical Naive Bayes Parameters are : {'alpha': 1.6}
The best score is : 0.49497777777777785


In [235]:
#SVM_(SVC) Grid

from sklearn.svm import SVC